In [3]:
import os
import sys
from pathlib import Path

# Lock working directory to main project root (2 levels up from src/notebook/)
current_dir = Path.cwd()
if current_dir.name == "notebook" and current_dir.parent.name == "src":
    project_root = current_dir.parent.parent
    os.chdir(project_root)
elif current_dir.name == "src":
    project_root = current_dir.parent
    os.chdir(project_root)
else:
    project_root = current_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Kernel working directory locked to:", Path.cwd())

Kernel working directory locked to: /home/s2025ae05884/workspace/heart-disease-predict


In [4]:
%%writefile src/preprocess.py
# src/preprocess.py
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

Writing src/preprocess.py


In [5]:

%%writefile -a src/preprocess.py
# Categorizing the 13 features of the UCI Heart Disease dataset
NUMERICAL_FEATURES = ["age", "trestbps", "chol", "thalach", "oldpeak"]
CATEGORICAL_FEATURES = ["cp", "restecg", "slope", "ca", "thal"]
BINARY_FEATURES = ["sex", "fbs", "exang"]

ALL_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES + BINARY_FEATURES
TARGET_COL = "target"

def build_preprocessor() -> ColumnTransformer:
    """
    Creates a reproducible Scikit-Learn ColumnTransformer that scales continuous
    features, one-hot encodes multi-category clinical variables, and imputes any
    unexpected missing values during production inference.
    """
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    binary_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, NUMERICAL_FEATURES),
            ("cat", categorical_transformer, CATEGORICAL_FEATURES),
            ("bin", binary_transformer, BINARY_FEATURES),
        ],
        remainder="drop"
    )
    return preprocessor

Appending to src/preprocess.py


In [7]:
import os
import shutil
import pandas as pd
from src.preprocess import build_preprocessor, ALL_FEATURES

target_path = "data/processed/heart_disease_cleaned.csv"

# 1. If the file was accidentally saved inside src/data/ or src/notebook/data/, copy it to root data/
possible_old_paths = [
    "src/data/processed/heart_disease_cleaned.csv",
    "src/notebook/data/processed/heart_disease_cleaned.csv"
]

if not os.path.exists(target_path):
    os.makedirs("data/processed", exist_ok=True)
    for old_path in possible_old_paths:
        if os.path.exists(old_path):
            shutil.copy(old_path, target_path)
            print(f"Copied dataset from '{old_path}' -> '{target_path}'")
            break

# 2. If it still doesn't exist anywhere, run acquire_and_clean_data() to create it in root data/
if not os.path.exists(target_path):
    from src.download_data import acquire_and_clean_data
    acquire_and_clean_data()

# 3. Now load and test the preprocessor!
df = pd.read_csv(target_path)
preprocessor = build_preprocessor()
X_trans = preprocessor.fit_transform(df[ALL_FEATURES])
print("Success! Loaded from:", os.path.abspath(target_path))
print("Original shape:", df[ALL_FEATURES].shape)
print("Transformed shape:", X_trans.shape)

Copied dataset from 'src/data/processed/heart_disease_cleaned.csv' -> 'data/processed/heart_disease_cleaned.csv'
Success! Loaded from: /home/s2025ae05884/workspace/heart-disease-predict/data/processed/heart_disease_cleaned.csv
Original shape: (303, 13)
Transformed shape: (303, 25)
